# Exploratory Data Analysis

Bucharest apartment listings from March 2019. The goal of this notebook is to understand the data before modeling: its quality, the distribution of prices, and which columns are related to price.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/raw/bucharest_house_prices.csv")

## First look

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

## Data quality checks

Column names are translated to English first.

In [ ]:
df = df.rename(columns={
    "Nr Camere": "rooms",
    "Suprafata": "area",
    "Etaj": "floor",
    "Total Etaje": "total_floors",
    "Sector": "sector",
    "Scor": "score",
    "Pret": "price",
})
df.columns

In [ ]:
# Fully identical rows (likely the same listing posted twice)
df.duplicated().sum()

In [ ]:
# Logically impossible rows: apartment floor above the building height
(df["floor"] > df["total_floors"]).sum()

In [ ]:
df["floor"].value_counts().sort_index()

In [ ]:
df["rooms"].value_counts().sort_index()

## Price distribution

In [ ]:
sns.histplot(df["price"], bins=50)
plt.title("Price distribution")
plt.xlabel("Price (EUR)")
plt.ylabel("Number of apartments")
plt.show()

## What drives the price

Duplicates are removed before looking at relationships.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
df.shape

In [ ]:
sns.scatterplot(data=df, x="area", y="price", alpha=0.3)
plt.title("Price vs area")
plt.xlabel("Area (sqm)")
plt.ylabel("Price (EUR)")
plt.show()

In [ ]:
# Price per square meter is used for exploration only.
# It is derived from the target, so it must never be used as a feature.
df["price_per_sqm"] = df["price"] / df["area"]

df.groupby("sector")["price_per_sqm"].median().round(0)

In [ ]:
df.groupby("score")["price_per_sqm"].median().round(0)

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation between columns")
plt.show()

### Does the floor position matter?

Correlation only captures linear relationships, so ground-floor and top-floor apartments are checked separately.

In [ ]:
df["floor_position"] = "middle"
df.loc[df["floor"] <= 0, "floor_position"] = "ground floor / basement"
df.loc[df["floor"] == df["total_floors"], "floor_position"] = "top floor"

df.groupby("floor_position")["price_per_sqm"].agg(["median", "count"]).round(0)

## EDA conclusions

- 3,529 rows, no missing values; 175 duplicates removed
- Price is right-skewed (median €78,600, max €495,000)
- Area is the strongest predictor (correlation 0.81)
- `rooms` is highly correlated with `area` (0.81)
- `score`: 1 = most expensive zone; two groups (1–2 and 3–5)
- `sector`: a category, not a quantity; Sector 1 is the most expensive
- Price variance grows with area
- Ground-floor and top-floor apartments are slightly cheaper per sqm (about 6% and 3%)